# 🐉 Forja de Contos — DPO no Colab (2ª tentativa)

Refina os adaptadores LoRA **SFT mais recentes** (os que têm o modo *família de NPCs*) com **DPO** e compara **base × SFT × DPO**.
O zip enviado na célula 3 precisa conter esses adaptadores novos (`adapter/pt` e `adapter/en`).

Mudanças em relação à 1ª tentativa (que não melhorou o modelo): **pares equilibrados** (~20 por idioma e modo, em vez de
concentrados em rumores), **toque leve** (taxa de aprendizado 1e-5, 1 época) e avaliação com **32 amostras** por pedido,
para reduzir o ruído da comparação.

Passos: gerar pares (boa × ruim) → treinar DPO em PT e EN → avaliar. Estimativa total: 40–70 min na T4 (a geração dos
pares é a parte mais lenta; se a sessão cair, rode a mesma célula com `--resume`).

**Antes de começar:** *Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU.*

## 1. Conferir a GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "Sem GPU! Ambiente de execução -> Alterar tipo -> T4 GPU"
print(torch.cuda.get_device_name(0))

## 2. Instalar dependências

In [ ]:
!pip install -q transformers peft datasets accelerate
# o Colab traz um torchao antigo que o peft novo recusa; o projeto não usa torchao
!pip uninstall -y -q torchao

## 3. Enviar o projeto
Selecione o arquivo **forja-de-contos-dpo.zip**.

In [ ]:
from google.colab import files
up = files.upload()
name = next(iter(up))
!unzip -o -q "{name}" -d /content/forja
%cd /content/forja
!rm -rf adapter_dpo results data/pairs.jsonl
!ls . adapter

## 4. Dataset de validação (para a perplexidade da avaliação)
Mesmo comando do SFT (com os poemas), para a validação ser idêntica e os números comparáveis.

In [ ]:
!python data/build_dataset.py --scale 2 --poems data/poems/portuguese-poems.csv

## 5. Gerar os pares de preferência (equilibrados)
O modelo SFT escreve 8 versões de cada pedido em temperatura alta; o `checks.py` separa as que passam das que falham.
A célula continua sorteando pedidos de cada modo até juntar ~20 pares (no máximo 120 pedidos por modo), para nenhum modo
dominar o treino. Modos em que o SFT quase não erra (ex.: missão em inglês) podem ficar abaixo de 20: tudo bem.
Se a sessão cair, rode a célula de novo acrescentando `--resume`.

In [ ]:
!python make_pairs.py --target-pairs 20 --max-prompts 120 --k 8

## 6. Treinar o DPO em português
Toque leve: taxa de aprendizado 1e-5 e 1 época. Acompanhe `acerto` e `margem` da validação: devem subir, mas pouco.
Se a margem ficar praticamente em zero (o ajuste não mexeu em nada), rode com `--lr 2e-5`.

In [ ]:
!python train_dpo.py --lang pt

## 7. Treinar o DPO em inglês

In [ ]:
!python train_dpo.py --lang en

## 8. Avaliar: base × ajustado (SFT) × dpo
Gera `results/eval.md` com perplexidade, **% de formato correto** (32 amostras por pedido) e diversidade das três versões.
Compare principalmente: o DPO subiu o formato em relação ao SFT **sem** derrubar a diversidade (distinct-n)?

In [ ]:
!python evaluate.py --n-format 32

In [ ]:
print(open("results/eval.md", encoding="utf-8").read())

In [ ]:
print(open("results/samples.md", encoding="utf-8").read()[:7000])

## 9. Baixar o resultado
`resultado_dpo.zip` com `adapter_dpo/`, `results/` e os pares usados.

In [ ]:
!zip -r -q resultado_dpo.zip adapter_dpo results data/pairs.jsonl
from google.colab import files
files.download("resultado_dpo.zip")